# Mass Euchre Game Runner

This notebook provides tools to run large numbers of euchre games for:
- Training data generation
- AI performance evaluation
- Strategy analysis
- Statistical analysis

## Setup and Imports

In [ ]:
import sys
import os
import json
from pathlib import Path

# Add the parent directory to the path to import euchre modules
sys.path.append(str(Path.cwd().parent))

from euchre.mass_game_runner import MassGameRunner
from euchre.ai_profiles import AggressiveAI, ConservativeAI, BalancedAI, OpportunisticAI
from euchre.models import PlayerType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

print("✅ Imports successful")

## Initialize Mass Game Runner

In [ ]:
# Initialize the mass game runner
runner = MassGameRunner()

print("🚀 Mass Game Runner initialized")

## View Available Team Configurations

In [ ]:
# The MassGameRunner has predefined team configurations
print("📋 Available team configurations:")
for config_name in runner.team_configs.keys():
    config = runner.team_configs[config_name]
    team1_players = config['team1']
    team2_players = config['team2']
    print(f"  {config_name}: {len(team1_players)} vs {len(team2_players)} players")
    print(f"    Team1: {[f'{p[0]} ({p[1]}, risk={p[2]})' for p in team1_players]}")
    print(f"    Team2: {[f'{p[0]} ({p[1]}, risk={p[2]})' for p in team2_players]}")

## Run Mass Games

In [ ]:
# Run games for each configuration
games_per_config = 100  # Adjust as needed
config_names = list(runner.team_configs.keys())
total_games = len(config_names) * games_per_config

print(f"🎮 Running {total_games} total games...")
print(f"📊 {games_per_config} games per configuration")
print(f"🎯 Configurations: {config_names}")

# Run games for each configuration
for config_name in config_names:
    print(f"\n🎯 Running {config_name}...")
    
    # Run games for this configuration using the correct API
    runner.run_games(games_per_config, config_name)
    
    print(f"  ✅ Completed {games_per_config} games for {config_name}")

print("\n✅ All games completed!")
print(f"📁 Results saved to: {runner.output_dir}")

## Load and Analyze Results

In [ ]:
# Get all game result files
game_files = runner.get_game_files()
print(f"📊 Found {len(game_files)} game result files")

# Load results into a list
all_games = []
for game_file in game_files:
    try:
        with open(game_file, 'r') as f:
            game_data = json.load(f)
            if 'error' not in game_data:  # Skip failed games
                all_games.append(game_data)
    except Exception as e:
        print(f"⚠️  Error loading {game_file}: {e}")

print(f"✅ Successfully loaded {len(all_games)} completed games")

## Convert to DataFrame

In [ ]:
# Convert results to DataFrame for analysis
if all_games:
    # Extract key data for analysis
    analysis_data = []
    for game in all_games:
        analysis_data.append({
            'game_id': game['game_id'],
            'config': game['team_config'],
            'winner': game['winner'],
            'team1_score': game['team1']['final_score'],
            'team2_score': game['team2']['final_score'],
            'total_rounds': game['total_rounds'],
            'game_duration': game['game_duration']
        })
    
    df = pd.DataFrame(analysis_data)
    print(f"📊 Total games analyzed: {len(df)}")
    print(f"\n📋 Columns: {list(df.columns)}")
    df.head()
else:
    print("⚠️  No completed games found to analyze")

## Statistical Analysis

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Overall statistics
    print("📊 OVERALL STATISTICS")
    print("=" * 50)
    
    # Win rates by configuration
    print("\n🏆 Win Rates by Configuration:")
    for config_name in df['config'].unique():
        config_df = df[df['config'] == config_name]
        team1_wins = len(config_df[config_df['winner'] == 'team1'])
        team2_wins = len(config_df[config_df['winner'] == 'team2'])
        ties = len(config_df[config_df['winner'] == 'tie'])
        total = len(config_df)
        
        print(f"  {config_name}:")
        print(f"    Team1: {team1_wins}/{total} ({team1_wins/total:.1%})")
        print(f"    Team2: {team2_wins}/{total} ({team2_wins/total:.1%})")
        if ties > 0:
            print(f"    Ties: {ties}/{total} ({ties/total:.1%})")
    
    # Game length statistics
    print(f"\n⏱️  Game Length Statistics:")
    print(f"  Average: {df['total_rounds'].mean():.1f} rounds")
    print(f"  Median: {df['total_rounds'].median():.1f} rounds")
    print(f"  Min: {df['total_rounds'].min()} rounds")
    print(f"  Max: {df['total_rounds'].max()} rounds")
    print(f"  Std Dev: {df['total_rounds'].std():.1f} rounds")
    
    # Score statistics
    print(f"\n📈 Score Statistics:")
    print(f"  Team1 Avg Score: {df['team1_score'].mean():.1f}")
    print(f"  Team2 Avg Score: {df['team2_score'].mean():.1f}")
    print(f"  Highest Score: {max(df['team1_score'].max(), df['team2_score'].max())}")
else:
    print("⚠️  No data available for analysis")

## Visualizations

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Set up plotting style
    plt.style.use('seaborn-v0_8')
    sns.set_palette("husl")
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Win rates by configuration
    ax1 = axes[0, 0]
    config_names = df['config'].unique()
    team1_win_rates = []
    team2_win_rates = []
    
    for config_name in config_names:
        config_df = df[df['config'] == config_name]
        team1_wins = len(config_df[config_df['winner'] == 'team1'])
        team2_wins = len(config_df[config_df['winner'] == 'team2'])
        total = len(config_df)
        
        team1_win_rates.append(team1_wins / total)
        team2_win_rates.append(team2_wins / total)
    
    x = range(len(config_names))
    width = 0.35
    
    ax1.bar([i - width/2 for i in x], team1_win_rates, width, label='Team1', alpha=0.8)
    ax1.bar([i + width/2 for i in x], team2_win_rates, width, label='Team2', alpha=0.8)
    ax1.set_xlabel('Configuration')
    ax1.set_ylabel('Win Rate')
    ax1.set_title('Win Rates by Configuration')
    ax1.set_xticks(x)
    ax1.set_xticklabels(config_names, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Game length distribution
    ax2 = axes[0, 1]
    ax2.hist(df['total_rounds'], bins=20, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Game Length (Rounds)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Game Length Distribution')
    ax2.grid(True, alpha=0.3)
    
    # 3. Win rates comparison
    ax3 = axes[1, 0]
    overall_team1_wins = len(df[df['winner'] == 'team1'])
    overall_team2_wins = len(df[df['winner'] == 'team2'])
    overall_ties = len(df[df['winner'] == 'tie'])
    total_games = len(df)
    
    labels = ['Team1 Wins', 'Team2 Wins']
    sizes = [overall_team1_wins, overall_team2_wins]
    colors = ['lightblue', 'lightcoral']
    
    if overall_ties > 0:
        labels.append('Ties')
        sizes.append(overall_ties)
        colors.append('lightgreen')
    
    ax3.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax3.set_title('Overall Win Distribution')
    
    # 4. Score distributions
    ax4 = axes[1, 1]
    ax4.hist(df['team1_score'], bins=15, alpha=0.7, label='Team1 Scores', density=True)
    ax4.hist(df['team2_score'], bins=15, alpha=0.7, label='Team2 Scores', density=True)
    ax4.set_xlabel('Final Score')
    ax4.set_ylabel('Density')
    ax4.set_title('Score Distributions')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available for visualization")

## Export Results

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Export results to CSV
    output_file = "mass_game_results.csv"
    df.to_csv(output_file, index=False)
    print(f"💾 Results exported to {output_file}")
    
    # Export summary statistics
    summary_stats = {
        'total_games': len(df),
        'team1_wins': len(df[df['winner'] == 'team1']),
        'team2_wins': len(df[df['winner'] == 'team2']),
        'ties': len(df[df['winner'] == 'tie']),
        'avg_game_length': df['total_rounds'].mean(),
        'avg_team1_score': df['team1_score'].mean(),
        'avg_team2_score': df['team2_score'].mean()
    }
    
    print("\n📊 Summary Statistics:")
    for key, value in summary_stats.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")
else:
    print("⚠️  No data available for export")

## Run All Configurations at Once

In [ ]:
# Alternative: Run all configurations at once
print("🚀 Running all configurations at once...")
games_per_config = 50  # Adjust as needed

runner.run_all_configurations(games_per_config)

print("✅ All configurations completed!")
print(f"📁 Results saved to: {runner.output_dir}")

## Cleanup Old Games

In [ ]:
# Clean up old game files (older than 7 days)
print("🧹 Cleaning up old game files...")
removed_count = runner.cleanup_old_games(7)
print(f"✅ Cleaned up {removed_count} old game files")

# Show current game count
current_count = runner.get_game_count()
print(f"📊 Current game files: {current_count}")